# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and processing the [FAIR<sup>2</sup> Mass Spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/) using the `mlcroissant` library following the [Croissant](https://mlcommons.org/croissant) schema. All references to record sets, fields, and columns use their `@id` values for clarity and reproducibility.

### Dataset Source
The dataset metadata is provided by the Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```


In [ ]:
# Ensure mlcroissant is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
ds_metadata = dataset.metadata

print(f"Dataset Name: {ds_metadata.name}\n")
print("Description:")
print(ds_metadata.description)

## 2. Data Overview
Here we list the available record sets in the dataset, then enumerate the fields and columns in each set, referencing them by their unique `@id`.

Note: All references use `@id` values per best practice.

In [ ]:
# Helper to safely extract @id from mlcroissant entities
def get_id(obj):
    return getattr(obj, 'id', None) or getattr(obj, '@id', None) or obj.get('@id', None)

print("Available record sets (by @id):")
record_sets = []
for rec_set in dataset.metadata.record_sets:
    rec_set_id = get_id(rec_set)
    record_sets.append(rec_set_id)
    print(f"  RecordSet @id: {rec_set_id}")
    print(f"    Name: {getattr(rec_set, 'name', '')}")
    # List fields
    if hasattr(rec_set, 'fields') and rec_set.fields:
        for field in rec_set.fields:
            print(f"      Field @id: {get_id(field)} | name: {getattr(field, 'name', '')} | dataType: {getattr(field, 'data_type', '')}")
            # List columns if any
            if hasattr(field, 'columns') and field.columns:
                for column in field.columns:
                    print(f"        Column @id: {get_id(column)} | name: {getattr(column, 'name', '')}")
print("\n")

## 3. Data Extraction
We load all record sets by their `@id` into `pandas` DataFrames for exploration. Replace the list below with the exact list of record set `@id`s from the previous section as appropriate for your dataset.

In [ ]:
# List all record set @id's discovered above (manually or parsed)
record_set_ids = record_sets # Auto-filled from above, but you may hard-code as needed

all_dataframes = dict()
for rsid in record_set_ids:
    print(f"Loading records for RecordSet @id='{rsid}' ...", end=" ")
    try:
        records = list(dataset.records(record_set=rsid))
        if records:
            df = pd.DataFrame(records)
            all_dataframes[rsid] = df
            print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}")
        else:
            print("No records found.")
    except Exception as e:
        print(f"ERROR: {e}")

# For demonstration, pick the first record set with non-empty DataFrame
example_record_set_id = None
for rsid, df in all_dataframes.items():
    if len(df):
        example_record_set_id = rsid
        break

print(f"\nExample DataFrame loaded for RecordSet @id: {example_record_set_id}")
print(all_dataframes[example_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

We'll demonstrate common EDA operations such as filtering, normalization, and grouping using one of the numeric fields in the chosen record set. All operations are based on `@id` referencing. Please update the field variables as needed based on the dataset's fields overview.

In [ ]:
# Example: Pick a numeric field from the DataFrame for analysis
df = all_dataframes[example_record_set_id]
print(f"Columns: {df.columns.tolist()}")

# Choose a numeric field; update this if necessary after review
numeric_field_id = None
for col in df.columns:
    # Heuristically pick the first float or int field (or by known field name)
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break
if not numeric_field_id:
    # Try to locate common field IDs by substring
    for col in df.columns:
        if 'abundance' in col.lower() or 'count' in col.lower() or 'mw' in col.lower() or 'coverage' in col.lower():
            numeric_field_id = col
            break

print(f"\nUsing numeric field for demo: {numeric_field_id}\n")

# Set a threshold for filtering
threshold = df[numeric_field_id].mean() if numeric_field_id else 0
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}, count = {len(filtered_df)}:")
display(filtered_df.head())

# Normalization of the field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()

print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a categorical field if any (e.g., 'Sample' or first non-numeric column)
group_field = None
for col in df.columns:
    if not pd.api.types.is_numeric_dtype(df[col]) and col != numeric_field_id:
        group_field = col
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"\nGrouped data by {group_field} (showing means):")
    display(grouped_df.head())
else:
    print("No categorical field found for grouping.")

## 5. Visualization
Here's an example of visualizing the distribution of the selected numeric field and, if available, comparing across groupings.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id], kde=True, bins=30)
plt.title(f"Distribution of '{numeric_field_id}'")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# Boxplot by group if group_field detected
if group_field:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field, y=numeric_field_id, data=filtered_df)
    plt.title(f"Distribution of '{numeric_field_id}' by '{group_field}'")
    plt.xticks(rotation=45)
    plt.show()


## 6. Conclusion
In this notebook, we've used the `mlcroissant` library to:

- Load and validate metadata from a FAIR Croissant schema
- Review available record sets, fields, and their `@id`s
- Extract and explore records programmatically by their Croissant identifiers
- Perform common EDA tasks such as filtering, normalization, and grouping
- Visualize data distributions for key numeric fields

This approach enables reproducible, standards-based data science workflows interoperable across repositories. For further analysis, you can select other record sets or fields by their `@id` and extend the EDA and modeling as required.